# Fraud Pattern Evolution & Anomaly Detection Framework

**Dataset:** 300 Financial Transaction Records from Kaggle Credit Card & Transaction Fraud Benchmark Dataset.  
**Objective:** Pemodelan analitik deteksi anomali transaksi (*Anomaly Detection Analytics*), klasterisasi perilaku penipuan (*Fraud Behavior Clustering*), dan pelacakan evolusi pola penipuan lintas saluran (*POS, E-Commerce, ATM, Mobile Banking*).

### Stage Breakdown:
1. **Data Preprocessing & Cleaning**: Pemuatan data transaksi keuangan Kaggle dan verifikasi integritas fitur anomali.
2. **Anomaly Score Indexing**: Perhitungan *Anomaly Score* dan indikator risiko penipuan (*Fraud Flag*).
3. **Publication-Grade EDA**: Visualisasi terpisah 300 DPI (Nilai Transaksi vs Skor Anomali, Kasus Penipuan Saluran, Matriks Anomali Spasial, dan Kurva Densitas Nominal).
4. **Econometric Fraud Modeling**: Model Regresi OLS determinan penentu skor anomali transaksi.
5. **Operational Risk Mitigation**: Export figure 300 DPI ke images/.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.formula.api import ols

os.makedirs('images', exist_ok=True)
os.makedirs('data', exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'figure.titlesize': 14,
    'figure.dpi': 300,
    'savefig.dpi': 300
})

print('Environment setup complete. Output directory ready.')

In [ ]:
df = pd.read_csv('data/kaggle_fraud_dataset.csv')
print('--- Dataset Shape ---')
print(df.shape)
print('\n--- Head Data ---')
print(df.head())

In [ ]:
# Model Regresi OLS Determinansi Skor Anomali Transaksi
model = ols('anomaly_score ~ amount_usd + is_fraud', data=df).fit()
print(model.summary())

In [ ]:
# 1. Transaction Amount vs Anomaly Score
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    df['amount_usd'],
    df['anomaly_score'],
    c=df['is_fraud'],
    cmap='coolwarm',
    alpha=0.75,
    s=60,
    edgecolors='black',
    linewidth=0.5
)
cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label('Fraud Flag (0 = Legitimate, 1 = Fraudulent)', fontsize=9)
ax.set_title('Transaction Amount vs Anomaly Score (Kaggle Fraud Benchmark)', fontweight='bold', pad=12)
ax.set_xlabel('Transaction Amount ()')
ax.set_ylabel('Anomaly Score (0-1)')
ax.axhline(df['anomaly_score'].mean(), color='#b91c1c', linestyle='--', alpha=0.8, label='Mean Anomaly Score')
ax.legend(loc='upper right', frameon=True)
plt.tight_layout()
plt.savefig('images/anomaly_score_vs_amount.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 2. Total Detected Fraud Cases Across Transaction Channels
fig, ax = plt.subplots(figsize=(8, 5))
channel_summary = df.groupby('channel')['is_fraud'].sum().sort_values(ascending=True)
bars = ax.barh(channel_summary.index, channel_summary.values, color='#1e3a8a', height=0.55, edgecolor='black', linewidth=0.5)
ax.set_title('Total Detected Fraud Cases Across Transaction Channels', fontweight='bold', pad=12)
ax.set_xlabel('Total Fraud Cases')
ax.grid(axis='y')

for bar in bars:
    width = bar.get_width()
    ax.text(width + 1, bar.get_y() + bar.get_height()/2, f'{int(width)} Cases', ha='left', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('images/fraud_cases_by_channel.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 3. Mean Anomaly Score Matrix (Geography vs Transaction Channel)
fig, ax = plt.subplots(figsize=(8, 6))
pivot_heatmap = df.pivot_table(index='geography', columns='channel', values='anomaly_score', aggfunc='mean')
sns.heatmap(pivot_heatmap, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax, cbar=True, linewidths=0.5)
ax.set_title('Mean Anomaly Score Matrix (Geography vs Transaction Channel)', fontweight='bold', pad=12)
ax.set_xlabel('Transaction Channel')
ax.set_ylabel('Geographic Region')
plt.tight_layout()
plt.savefig('images/anomaly_matrix_geography_channel.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 4. Transaction Amount Probability Density Function (Legitimate vs Fraud)
fig, ax = plt.subplots(figsize=(8, 5))
sns.kdeplot(df[df['is_fraud'] == 0]['amount_usd'], fill=True, label='Legitimate (Class 0)', ax=ax, color='#2563eb', alpha=0.3)
sns.kdeplot(df[df['is_fraud'] == 1]['amount_usd'], fill=True, label='Fraudulent (Class 1)', ax=ax, color='#dc2626', alpha=0.3)
ax.set_title('Transaction Amount Probability Density Function (Legitimate vs Fraud)', fontweight='bold', pad=12)
ax.set_xlabel('Transaction Amount ()')
ax.set_ylabel('Probability Density')
ax.legend(loc='upper right', frameon=True)
plt.tight_layout()
plt.savefig('images/amount_density_fraud_vs_legit.png', dpi=300, bbox_inches='tight')
plt.show()